<a href="https://colab.research.google.com/github/ProfessorPatrickSlatraigh/cis9557__baseline/blob/main/CIS9557_VADER_Sentiment_Demo_v03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sentiment Analysis with VADER
### CIS9557 — Business Analytics
<i>by Professor Patrick: Mar-2026</i>
  
<i>a copy of this notebook is available at [bit.ly/cis9557SentimentAnalysisVADER](https://bit.ly/cis9557SentimentAnalysisVADER)</i>

---

## Purpose

This notebook demonstrates how to apply the **VADER** (Valence Aware Dictionary and sEntiment Reasoner) lexicon tool from Python's `nltk` library to evaluate sentiment in a block of text.

VADER is a rule-based sentiment analyzer designed for text that appears in customer reviews, news articles, social media posts, and other short-form written expression. It requires no training data and no statistical modeling. It works by matching words against a pre-built dictionary of terms with assigned sentiment weights and applying a set of grammatical heuristics to adjust for context.

VADER produces four scores for any input string:

| Score | Meaning |
|:---|:---|
| `neg` | Proportion of text with negative valence (0.0 to 1.0) |
| `neu` | Proportion of text with neutral valence (0.0 to 1.0) |
| `pos` | Proportion of text with positive valence (0.0 to 1.0) |
| `compound` | Normalized aggregate score ranging from -1.0 to +1.0 |

The `compound` score is the primary metric for overall classification:

| Compound Value | Classification |
|:---|:---|
| >= 0.05 | **Positive** |
| <= -0.05 | **Negative** |
| Between -0.05 and 0.05 | **Neutral** |

---

## How to Use This Notebook

Work through the cells in order from top to bottom. Each section is preceded by an explanation of what the code does and why. Steps 1 through 7 are the instructional walkthrough. Step 8 contains a reusable function you can use to repeat the full analysis on any new text without re-running each cell individually.

Run each cell by clicking on it and pressing **Shift + Enter**.

---
## Step 1 — Install and Import Dependencies

Before any analysis can run, the required Python libraries must be installed and imported. This notebook depends on three libraries:

- **`nltk`** (Natural Language Toolkit) provides the VADER analyzer and the sentence tokenizer used in Step 4.
- **`pandas`** is used to organize sentence-level results into a structured table.
- **`matplotlib`** is used to produce the bar chart in Step 6.

The first cell below installs `nltk` if it is not already present in your Python environment. The second cell imports all three libraries and downloads two NLTK data files: the VADER lexicon and the sentence tokenizer model. Both data files are required before the analyzer can be used.

Both cells must complete successfully before you proceed. You should see a confirmation message printed at the bottom of each cell after execution.

In [ ]:
# Install nltk if it is not already present in this environment.
# After the first successful run, this cell can be skipped or commented out.
import subprocess
subprocess.run(['pip', 'install', 'nltk', '--quiet'], check=True)
print('nltk installation confirmed.')

The cell below imports all required libraries and downloads the necessary NLTK data files. The `quiet=True` parameter suppresses verbose download output. If the data files have already been downloaded to your environment, NLTK skips the download automatically and proceeds without error.

In [ ]:
import nltk
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.tokenize import sent_tokenize

# Download required NLTK data files
nltk.download('vader_lexicon', quiet=True)
nltk.download('punkt',         quiet=True)
nltk.download('punkt_tab',     quiet=True)

print('All dependencies loaded successfully.')

---
## Step 2 — Define the Text Variable

The text to be analyzed is stored as a **docstring variable**: a triple-quoted Python string that can span multiple lines without requiring explicit line continuation characters. This format is used because real-world source text, such as the body of a news article or a customer review, almost always contains line breaks and multiple paragraphs.

The variable is named `sample_text`. A demonstration passage has been provided so you can run the full notebook on the first pass without substituting your own content. In Step 8 you will paste your own article text into a separate variable.

The cell below also prints a character count and a preview of the loaded text so you can confirm it was assigned correctly before proceeding to the analysis.

In [ ]:
sample_text = """
The service at this location has improved dramatically over the past six months.
Staff members are attentive, knowledgeable, and genuinely helpful.
The wait time was slightly longer than expected on a Saturday afternoon,
but the quality of the product more than compensated for the delay.
I would not hesitate to recommend this business to colleagues and family.
One minor complaint: the parking situation remains frustrating and poorly organized.
Overall, however, this was an excellent experience.
"""

print('Text loaded. Character count:', len(sample_text))
print('\n--- Text Preview ---')
print(sample_text.strip())

---
## Step 3 — Score the Full Text Block

The first scoring pass treats the entire docstring as a single unit and returns one set of four scores for the whole passage.

This is accomplished by creating an instance of `SentimentIntensityAnalyzer` and calling its `polarity_scores()` method with the text string as the argument. The method returns a Python dictionary with four keys: `neg`, `neu`, `pos`, and `compound`.

The second cell applies the standard VADER classification thresholds to the `compound` score and prints the overall sentiment label. This aggregate score is a useful starting point, but it conceals variation within the text. A passage that contains both strongly positive and strongly negative sentences may still produce a near-neutral compound score if the opposing signals cancel each other out. Step 4 addresses this limitation by scoring each sentence individually.

In [ ]:
# Instantiate the VADER analyzer
sia = SentimentIntensityAnalyzer()

# Score the full text block as a single string
scores = sia.polarity_scores(sample_text)

print('--- VADER Scores for Full Text ---')
for label, value in scores.items():
    print(f'  {label:>10} : {value:.4f}')

The cell below reads the `compound` value from the scores dictionary and applies the standard VADER classification thresholds to assign a sentiment label. The thresholds (+0.05 and -0.05) were established by the original VADER researchers based on empirical validation and are the standard reference values used across the literature.

In [ ]:
compound = scores['compound']

if compound >= 0.05:
    classification = 'POSITIVE'
elif compound <= -0.05:
    classification = 'NEGATIVE'
else:
    classification = 'NEUTRAL'

print(f'Compound score : {compound:.4f}')
print(f'Classification : {classification}')

---
## Step 4 — Sentence-Level Breakdown

Scoring the full text as a single block produces one number for the entire passage, which is often too coarse for business interpretation. A passage that contains one strongly negative sentence surrounded by positive ones will yield an aggregate score that understates the negative signal. Sentence-level scoring makes each individual statement visible and independently interpretable.

This step uses NLTK's `sent_tokenize()` function to split the docstring into individual sentences based on punctuation patterns and capitalization heuristics. Each sentence is then passed separately to `polarity_scores()`, and the results are collected into a Pandas DataFrame with one row per sentence.

The first cell below confirms how many sentences were detected and prints each one so you can verify the tokenization before reviewing the scores. The second cell runs the scoring loop and displays the complete results table. The `Text` column is truncated to 80 characters for display readability; the full sentence text is preserved in the underlying DataFrame.

In [ ]:
# Split the text into individual sentences
sentences = sent_tokenize(sample_text.strip())

print(f'{len(sentences)} sentence(s) detected.\n')
for i, s in enumerate(sentences, 1):
    print(f'[{i}] {s}')

The cell below scores each sentence in turn, applies the classification thresholds to assign a label, and stores all results in a DataFrame named `df`. Review the `compound` column and the `Label` column to identify which sentences are pulling the overall score upward or downward.

In [ ]:
# Score each sentence and collect results in a DataFrame
rows = []
for i, sentence in enumerate(sentences, 1):
    sc = sia.polarity_scores(sentence)
    if sc['compound'] >= 0.05:
        label = 'Positive'
    elif sc['compound'] <= -0.05:
        label = 'Negative'
    else:
        label = 'Neutral'
    rows.append({
        'Sentence #' : i,
        'Text'       : sentence,
        'neg'        : sc['neg'],
        'neu'        : sc['neu'],
        'pos'        : sc['pos'],
        'compound'   : sc['compound'],
        'Label'      : label
    })

df = pd.DataFrame(rows)

# Truncate text column to 80 characters for display only
display_df = df.copy()
display_df['Text'] = display_df['Text'].str[:80] + display_df['Text'].apply(
    lambda x: '...' if len(x) > 80 else ''
)

display(display_df)

---
## Step 5 — Summary Statistics by Sentiment Label

The sentence-level table produced in Step 4 is detailed but not immediately easy to interpret at a glance when working with a long text. This step aggregates the results by sentiment label and computes descriptive statistics for each group: count of sentences, mean compound score, minimum compound score, and maximum compound score.

The cell also calculates what percentage of sentences fall into each category. Percentages are more meaningful for business comparison than raw counts when the texts being compared are of different lengths, such as a short review versus a long investigative article. A single strongly negative sentence in a ten-sentence passage represents ten percent of the content; the same sentence in a hundred-sentence article represents one percent.

In [ ]:
summary = (
    df.groupby('Label')
      .agg(
          Count         = ('Label',    'count'),
          Mean_Compound = ('compound', 'mean'),
          Min_Compound  = ('compound', 'min'),
          Max_Compound  = ('compound', 'max')
      )
      .round(4)
      .reset_index()
)

print('--- Summary Statistics by Sentiment Label ---')
display(summary)

total = len(df)
print()
for _, row in summary.iterrows():
    pct = 100 * row['Count'] / total
    print(f"  {row['Label']:>8}: {int(row['Count'])} sentence(s)  ({pct:.1f}%)")

---
## Step 6 — Visualize Compound Scores by Sentence

This step produces a bar chart that plots the compound score for each sentence. Bars are color-coded by classification label: teal for positive, red for negative, and grey for neutral. Two dashed horizontal reference lines mark the classification thresholds at +0.05 and -0.05.

The chart makes the distribution of sentiment across the passage immediately visible in a way that a table does not. A text where most bars are tall and teal is uniformly positive. A text with alternating colors signals mixed or inconsistent sentiment, which may reflect a publication presenting both sides of an issue, an article that shifts tone across sections, or a review that praises certain aspects of a product while criticizing others.

The chart is saved as a PNG file in the current working directory so it can be included in a report or presentation. If you run the analysis on multiple texts, rename the saved file after each run to avoid overwriting it.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

colors = df['Label'].map({
    'Positive': '#4CC9A8',
    'Negative': '#C94C4C',
    'Neutral' : '#AAAAAA'
})

ax.bar(df['Sentence #'], df['compound'],
       color=colors, edgecolor='white', linewidth=0.5)

ax.axhline(y= 0.05, color='#4CC9A8', linestyle='--', linewidth=0.8, alpha=0.7)
ax.axhline(y=-0.05, color='#C94C4C', linestyle='--', linewidth=0.8, alpha=0.7)
ax.axhline(y= 0,    color='#333333', linestyle='-',  linewidth=0.5, alpha=0.4)

ax.set_xlabel('Sentence Number', fontsize=11)
ax.set_ylabel('Compound Score',  fontsize=11)
ax.set_title('VADER Compound Sentiment Score by Sentence',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xticks(df['Sentence #'])
ax.set_ylim(-1.1, 1.1)

legend_patches = [
    mpatches.Patch(color='#4CC9A8', label='Positive'),
    mpatches.Patch(color='#AAAAAA', label='Neutral'),
    mpatches.Patch(color='#C94C4C', label='Negative'),
]
ax.legend(handles=legend_patches, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('vader_sentence_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to vader_sentence_scores.png')

---
## Step 7 — Business Interpretation

Sentiment scores become analytically useful only when they are connected to a specific business question. A compound score of 0.42 is not inherently meaningful; what matters is what that score represents in the context of the text and what a decision-maker should do with it.

This step produces two outputs that support that interpretive work.

The first cell identifies the single sentence with the highest positive compound score and the single sentence with the lowest (most negative) compound score. These are the statements that contributed most strongly to each direction of sentiment. In business applications, these often correspond to the clearest statement of praise or the sharpest expression of criticism in a review, article, or earnings call transcript.

The second cell prints a structured management summary combining the full-text compound score, the mean sentence score, and the percentage breakdown by label. This format is appropriate for a business briefing: a concise set of quantitative findings that a manager can assess without reading the full table. The interpretation note at the bottom is a placeholder; in a completed deliverable it would be replaced with specific observations about the text and its implications.

In [ ]:
# Identify the sentences with the strongest positive and negative signals
positive_sentences = df[df['Label'] == 'Positive'].sort_values('compound', ascending=False)
negative_sentences = df[df['Label'] == 'Negative'].sort_values('compound')

print('=== Strongest Positive Signal ===')
if not positive_sentences.empty:
    top_pos = positive_sentences.iloc[0]
    print(f'  Compound : {top_pos["compound"]:.4f}')
    print(f'  Text     : {top_pos["Text"]}')
else:
    print('  No positive sentences detected.')

print()
print('=== Strongest Negative Signal ===')
if not negative_sentences.empty:
    top_neg = negative_sentences.iloc[0]
    print(f'  Compound : {top_neg["compound"]:.4f}')
    print(f'  Text     : {top_neg["Text"]}')
else:
    print('  No negative sentences detected.')

The cell below prints the management summary. In a completed exercise submission, replace the placeholder interpretation text with specific observations about your source article: which sentences drove the result, whether the VADER output aligned with your own reading of the text, and what the findings would imply for a decision-maker monitoring coverage of the topic.

In [ ]:
overall_compound = scores['compound']
mean_sentence    = df['compound'].mean()
pct_positive     = 100 * (df['Label'] == 'Positive').sum() / len(df)
pct_negative     = 100 * (df['Label'] == 'Negative').sum() / len(df)

print('=== Management Summary ===')
print(f'  Overall compound score (full text) : {overall_compound:.4f}  ->  {classification}')
print(f'  Mean compound score (per sentence) : {mean_sentence:.4f}')
print(f'  Sentences classified Positive      : {pct_positive:.1f}%')
print(f'  Sentences classified Negative      : {pct_negative:.1f}%')
print()
print('[ Replace this block with your written interpretation. ]')
print('Identify which specific sentences drove the result.')
print('Note where the VADER classification agrees or disagrees')
print('with your own reading of the text, and explain what the')
print('findings would mean for a relevant business decision-maker.')

---
## Step 8 — Reusable Pipeline Function

Steps 1 through 7 walked through the analysis one operation at a time for instructional purposes. The function defined in the cell below, `analyze_sentiment()`, packages the complete pipeline into a single callable. Once this definition cell has been run, you can analyze any new text by editing the `new_text` variable in the execution cell at the bottom of this section and running one line.

The function accepts any Python string as its argument, including a multi-line triple-quoted docstring, and performs every step in sequence: full-text scoring, sentence tokenization, sentence-level scoring, summary statistics, strongest signal identification, and bar chart generation. It returns two objects so that additional analysis can be performed on the results if needed: the sentence-level DataFrame and the full-text scores dictionary.

Run the cell below now to register the function in memory. No output will appear; the function is only defined here, not executed.

In [ ]:
def analyze_sentiment(text):
    """
    Run a complete VADER sentiment analysis on the supplied text string.

    Performs the following steps in sequence:
        1. Scores the full text block as a single unit.
        2. Tokenizes the text into sentences and scores each individually.
        3. Prints the sentence-level results table.
        4. Prints a management summary: overall label, mean compound score,
           and percentage of sentences classified positive and negative.
        5. Identifies and prints the strongest positive and negative sentences.
        6. Displays a bar chart of compound scores by sentence.

    Parameters
    ----------
    text : str
        Any string or triple-quoted docstring to be analyzed.
        Paste the body text of an article, review, or other source
        directly into the triple-quoted variable in the execution cell.

    Returns
    -------
    df : pandas.DataFrame
        One row per sentence with columns:
        Sentence #, Text, neg, neu, pos, compound, Label.
    scores : dict
        VADER scores for the full text block
        (keys: neg, neu, pos, compound).
    """

    # Score the full text block
    sia    = SentimentIntensityAnalyzer()
    scores = sia.polarity_scores(text)

    compound = scores['compound']
    if compound >= 0.05:
        overall_label = 'POSITIVE'
    elif compound <= -0.05:
        overall_label = 'NEGATIVE'
    else:
        overall_label = 'NEUTRAL'

    # Sentence-level scoring
    sentences = sent_tokenize(text.strip())
    rows = []
    for i, sentence in enumerate(sentences, 1):
        sc = sia.polarity_scores(sentence)
        if sc['compound'] >= 0.05:
            label = 'Positive'
        elif sc['compound'] <= -0.05:
            label = 'Negative'
        else:
            label = 'Neutral'
        rows.append({
            'Sentence #' : i,
            'Text'       : sentence,
            'neg'        : sc['neg'],
            'neu'        : sc['neu'],
            'pos'        : sc['pos'],
            'compound'   : sc['compound'],
            'Label'      : label
        })
    df = pd.DataFrame(rows)

    # Display sentence-level results table
    display_df = df.copy()
    display_df['Text'] = display_df['Text'].str[:80] + display_df['Text'].apply(
        lambda x: '...' if len(x) > 80 else ''
    )
    print('--- Sentence-Level Results ---')
    display(display_df)

    # Management summary
    total        = len(df)
    mean_sent    = df['compound'].mean()
    pct_positive = 100 * (df['Label'] == 'Positive').sum() / total
    pct_negative = 100 * (df['Label'] == 'Negative').sum() / total

    print('\n=== Management Summary ===')
    print(f'  Overall compound score (full text) : {compound:.4f}  ->  {overall_label}')
    print(f'  Mean compound score (per sentence) : {mean_sent:.4f}')
    print(f'  Sentences classified Positive      : {pct_positive:.1f}%')
    print(f'  Sentences classified Negative      : {pct_negative:.1f}%')

    # Strongest signals
    pos_df = df[df['Label'] == 'Positive'].sort_values('compound', ascending=False)
    neg_df = df[df['Label'] == 'Negative'].sort_values('compound')

    print('\n=== Strongest Positive Signal ===')
    if not pos_df.empty:
        r = pos_df.iloc[0]
        print(f'  Compound : {r["compound"]:.4f}')
        print(f'  Text     : {r["Text"]}')
    else:
        print('  No positive sentences detected.')

    print('\n=== Strongest Negative Signal ===')
    if not neg_df.empty:
        r = neg_df.iloc[0]
        print(f'  Compound : {r["compound"]:.4f}')
        print(f'  Text     : {r["Text"]}')
    else:
        print('  No negative sentences detected.')

    # Bar chart
    colors = df['Label'].map({
        'Positive': '#4CC9A8',
        'Negative': '#C94C4C',
        'Neutral' : '#AAAAAA'
    })
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(df['Sentence #'], df['compound'],
           color=colors, edgecolor='white', linewidth=0.5)
    ax.axhline(y= 0.05, color='#4CC9A8', linestyle='--', linewidth=0.8, alpha=0.7)
    ax.axhline(y=-0.05, color='#C94C4C', linestyle='--', linewidth=0.8, alpha=0.7)
    ax.axhline(y= 0,    color='#333333', linestyle='-',  linewidth=0.5, alpha=0.4)
    ax.set_xlabel('Sentence Number', fontsize=11)
    ax.set_ylabel('Compound Score',  fontsize=11)
    ax.set_title('VADER Compound Sentiment Score by Sentence',
                 fontsize=13, fontweight='bold', pad=12)
    ax.set_xticks(df['Sentence #'])
    ax.set_ylim(-1.1, 1.1)
    legend_patches = [
        mpatches.Patch(color='#4CC9A8', label='Positive'),
        mpatches.Patch(color='#AAAAAA', label='Neutral'),
        mpatches.Patch(color='#C94C4C', label='Negative'),
    ]
    ax.legend(handles=legend_patches, loc='lower right', fontsize=9)
    plt.tight_layout()
    plt.show()

    return df, scores

---
## Step 8 (continued) — Substitute Your Own Text

### Instructions for the Group Exercise

Each group member should locate a **freely accessible** online article, news story, or commentary on the shared topic assigned to your group. Sources behind a paywall, such as the Wall Street Journal or the Financial Times, cannot be used for this exercise because their text cannot be copied without a subscription.

Once you have selected an article, follow these steps precisely before running the cell below.

**Step A — Copy the body text only.**
Select from the first sentence of the first paragraph to the last sentence of the final paragraph. Do not include the headline, the author byline, the publication date, image captions, pull quotes formatted separately from the body, or any navigation or footer text that appears on the page. Including non-body text will introduce noise into the sentence-level scores.

**Step B — Paste the copied text into the triple-quoted variable.**
Replace the placeholder text in `new_text` entirely. The triple quotes must remain in place; paste only between them.

**Step C — Record the source in the comment line.**
Fill in the comment line above `new_text` with the publication name, author (if shown), publication date, and URL. This documents your source alongside the analysis and makes your work reproducible.

**Step D — Run the cell.**
The full analysis will execute in a single step and produce the table, summary, strongest signals, and bar chart.

---

**Note on article length.** Articles longer than approximately 40 sentences will produce a bar chart where individual bars become too narrow to read clearly. If your article is long, copy only the first 20 to 25 sentences for this exercise and note in your comment line how much of the article you used. The analytical value of the exercise is not diminished by using a portion of the text.

---

### Discussion Questions for Group Comparison

After all group members have run their own articles on the same topic, compare results across sources and consider the following questions:

1. Does one publication express measurably more positive or negative sentiment on this topic than another? What might explain the difference?
2. Are there individual sentences where the VADER classification does not match your own reading of the text? What about the sentence caused the discrepancy?
3. Does the aggregate compound score tell the same story as the sentence-level breakdown, or do they diverge? When they diverge, which is more useful for business interpretation?
4. What would these sentiment differences mean for a business decision-maker or analyst monitoring coverage of this topic across multiple publications?

In [ ]:
# Source: [Publication name] | [Author, if shown] | [Date] | [URL]
# Portion used: [e.g., full article / first 20 sentences / paragraphs 1-5]

new_text = """
Paste the body text of your chosen article here.
Remove this placeholder text entirely before running the cell.
Do not include the headline, byline, date, or any text outside
the article body paragraphs.
"""

results_df, full_scores = analyze_sentiment(new_text)

---
## Notes and Limitations

The following limitations are important to understand before drawing conclusions from VADER output in a business context.

**1. VADER is lexicon-based, not learned.** It does not adapt to domain-specific vocabulary. Words that carry strong sentiment in financial reporting, medical writing, or legal text may not be weighted appropriately in the VADER lexicon, which was built and validated primarily on social media text.

**2. Sarcasm and irony are not detected.** VADER applies heuristic rules for simple negation (for example, "not good" is treated as negative) but does not identify sarcasm. A sentence such as "Great, another product recall" would likely score as positive because of the word "great."

**3. The compound score is not a probability.** A compound score of 0.80 does not mean the text is eighty percent positive. It is a normalized aggregate of valence weights, appropriate for ranking texts relative to each other and for threshold-based classification, but not for probabilistic inference.

**4. Sentence tokenization quality depends on punctuation.** Poorly punctuated text will produce inaccurate sentence boundaries. A long unpunctuated passage may be treated as a single sentence, collapsing the sentence-level analysis entirely. Cleaning and standardizing punctuation before analysis improves results.

**5. Source register affects scores independently of content.** Different publications use different editorial styles. A tabloid and a financial newspaper covering the same event will produce different sentiment profiles not because their underlying assessments differ but because their writing styles differ. Cross-source comparisons should account for register differences.

**6. VADER is a starting point, not a final answer.** In professional practice, VADER output is typically one input into a broader analytical process that includes human review of flagged passages, domain-specific lexicon augmentation, or a supervised classification model trained on labeled data from the relevant industry.

---
